# Análise Exploratória de Vendas de Videogames

**Disciplina:** Estatística | **Integrantes:**                                                                                                                                  Arthur Sindeaux de Araújo Nogueira
Guilherme Tolentino Leitão de Melo
Rafael Coutinho Lima
Lucas Pinto Ribeiro Beno
Pedro Coutinho da Silva
Arthur Oliveira Furieri
João Henrique Bastos
João Eduardo Azevedo de Andrade

## Problema principal

Analisar as vendas de títulos de videogame para entender como o desempenho comercial se distribui entre gêneros, plataformas e regiões.

## Perguntas de investigação e hipóteses

**Pergunta 1.** Como se distribui o desempenho comercial dos títulos, e quão concentrado está o mercado em poucos lançamentos?
*Hipótese 1:* a distribuição é fortemente assimétrica à direita, com média bastante superior à mediana e o terceiro quartil ainda em patamar baixo. Se confirmado, o jogo mediano vende uma fração do que a média sugere, e a média é uma métrica enganosa para planejar um lançamento.

**Pergunta 2.** O desempenho de um título em um mercado se repete nos demais, ou cada região tem dinâmica própria?
*Hipótese 2:* as receitas regionais não se movem juntas de forma uniforme. Esperamos divergência entre Pearson e Spearman, porque Pearson é dominado pelos poucos títulos gigantes enquanto Spearman descreve o jogo típico. Se as duas medidas apontarem direções diferentes, a previsibilidade entre mercados só existe na faixa dos grandes sucessos.

**Pergunta 3.** Gêneros e plataformas com mais lançamentos são também os de melhor desempenho típico?
*Hipótese 3:* volume de lançamentos não acompanha desempenho mediano. Esperamos que categorias mais exploradas apresentem mediana comprimida por saturação.

## Fonte dos dados

Dataset `vgsales.csv` (Video Game Sales, Kaggle), com estimativas de vendas em milhões de unidades. Cada linha representa **um jogo em uma plataforma específica**: o mesmo título lançado em dois consoles ocupa duas linhas.

# 1 Config do Ambiente

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np


In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip install pandas matplotlib numpy

## 2. Carga e reconhecimento inicial dos dados

In [ ]:
df = pd.read_csv("data/vgsales.csv")

print("Linhas e colunas:", df.shape)
df.head()
df.info()
df.describe()

**Leitura inicial.** A base tem 16.598 registros e 11 colunas. Já aparecem dois problemas a tratar: `Year` foi lida como decimal, o que é sintoma de valores ausentes, e as colunas de venda têm média muito distante do máximo, primeiro indício da assimetria que a Pergunta 1 investiga.

## 3. Pré-processamento


In [ ]:
nulos = pd.DataFrame({
    "ausentes": df.isnull().sum(),
    "percentual": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos[nulos["ausentes"] > 0]

**Diagnóstico de nulos.** Apenas duas colunas têm ausências e ambas em volume pequeno: `Year` (1,63%) e `Publisher` (0,35%). Nenhuma coluna de vendas tem valor faltante, o que importa porque as perguntas de investigação dependem exclusivamente delas. Antes de aplicar o tratamento, verificamos as duplicatas: parte das linhas repetidas tem justamente `Year` ausente, e é preciso saber quais casos estão saindo da base por esse critério.

### 3.1 Duplicatas

In [ ]:
print("Linhas idênticas em todas as colunas:", df.duplicated().sum())

dup = df[df.duplicated(subset=["Name", "Platform"], keep=False)]
print("Linhas com Name + Platform repetidos:", len(dup))
dup.sort_values(["Name", "Platform"])

**Leitura.** `duplicated()` sobre a linha inteira retorna zero, mas a chave lógica da base (título + plataforma) revela 10 linhas repetidas, de três naturezas distintas:

- **Não são duplicatas.** `Need for Speed: Most Wanted` aparece duas vezes em PC e duas em X360, com anos diferentes (2005 e 2012). São o original e o remake homônimo — jogos distintos, devem ser mantidos.
- **Duplicata verdadeira.** `Wii de Asobu: Metroid Prime` (Wii) aparece duas vezes com valores de venda idênticos. É erro de registro. Ambas as ocorrências têm `Year` ausente.
- **Registros fragmentados.** `Madden NFL 13` (PS3) e `Sonic the Hedgehog` (PS3) aparecem duas vezes com vendas diferentes, aparentemente por divisão no registro da fonte. A segunda linha do Sonic também tem `Year` ausente.

Três dessas linhas saem automaticamente no tratamento de nulos da próxima seção. Resta apenas o par do `Madden NFL 13`, que optamos por manter: somá-las exigiria supor a origem da fragmentação, e o impacto em 16.598 registros é irrelevante.

A conclusão metodológica é que `duplicated()` sem critério teria retornado zero e nos daria falsa segurança. Verificar pela **chave lógica** da base, e não pela linha inteira, foi o que revelou os casos reais.

### 3.2 Tratamento dos valores ausentes

Optamos por remover as linhas com valores ausentes em vez de imputar. `Year` representa 1,63% dos registros e não há como estimar um ano de lançamento sem inventar informação; `Publisher` são 0,35%. Somadas, a perda fica abaixo de 2% e nenhuma coluna de vendas é afetada, de modo que as medidas descritivas de `Global_Sales` permanecem íntegras.

In [ ]:
antes = len(df)
df = df.dropna()
print(f"Removidas {antes - len(df)} linhas com valores ausentes. Restam {len(df)} registros.")
print(f"Perda: {(antes - len(df)) / antes * 100:.2f}%")
print("Nulos restantes:", df.isnull().sum().sum())